# C1: Unsupervised Learning

## 1. Introduction to Unsupervised Learning

**Supervised Learning** involves training a model on a labeled dataset, where the algorithm learns to map inputs to known outputs (e.g., classifying emails as spam or not spam based on labeled examples).

**Unsupervised Learning**, on the other hand, deals with **unlabeled data**. The goal is to discover hidden patterns, structures, or relationships within the data without any prior knowledge of the outcomes. It's like a computer trying to learn by itself without a teacher.

### Key Concepts
1.  **Clustering**: Grouping similar data points together (e.g., K-Means, Hierarchical Clustering).
2.  **Dimensionality Reduction**: Reducing the number of variables in the data while preserving essential information (e.g., PCA).

![Clustering Example](https://upload.wikimedia.org/wikipedia/commons/c/c8/Cluster-2.svg)
*(Source: Wikimedia Commons - Example of clustering where similar data points are grouped together)*

## 2. K-Means Clustering

K-Means is one of the most popular clustering algorithms. It partitions the data into **K** distinct clusters.

### How it Works (The Algorithm)
1.  **Initialization**: Randomly select **K** data points as initial centroids (centers).
2.  **Assignment**: Assign every data point to the nearest centroid (usually based on Euclidean distance).
3.  **Update**: Recalculate the centroid of each cluster (the mean of all points in the cluster).
4.  **Repeat**: Repeat steps 2 and 3 until the centroids stop moving (convergence).

### Visualization of Convergence
Observe how the centroids move and the cluster assignments stabilize over iterations:

![K-Means Convergence Animation](https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif)
*(Source: Wikimedia Commons - K-means convergence animation showing iterative centroid updates)*

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# For visualization purposes
%matplotlib inline

### 3. Practical Implementation

We will use a dataset containing measurements of different wheat seeds. The goal is to cluster them based on their physical characteristics.

In [ ]:
# Load the dataset
df = pd.read_csv('../../00-data/seeds.csv', header=None, names=['area', 'perimeter', 'compactness', 'kernel_length', 'kernel_width', 'asymmetry', 'groove_length', 'variety'])

# Display the first few rows
print(df.head())

# Check for missing values
print(df.info())

### Preprocessing
K-Means is sensitive to the scale of the data because it uses distance calculations. It's crucial to scale the features so that one feature (e.g., weight) doesn't dominate another (e.g., length).

In [ ]:
# Select numerical features for clustering (dropping any potential non-numeric columns if they exist, though seeds.csv is usually all numeric)
# We'll use all columns for clustering
X = df.values

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### Training the Model
Let's start assuming we want 3 clusters ($K=3$).

In [ ]:
# Initialize KMeans with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42)

# Fit the model
kmeans.fit(X_scaled)

# Get the cluster labels
labels = kmeans.labels_

# Add labels to the original dataframe for analysis
df['Cluster'] = labels
print(df.head())

### Visualization
Visualizing high-dimensional data is hard, but we can plot two features against each other (e.g., Area vs Perimeter) and color by cluster.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['area'], df['perimeter'], c=df['Cluster'], cmap='viridis', alpha=0.6)
plt.xlabel('Area')
plt.ylabel('Perimeter')
plt.title('K-Means Clustering: Area vs Perimeter')
plt.colorbar(label='Cluster')
plt.show()

## 4. Evaluating Clusters: The Elbow Method

How do we know $K=3$ is the best choice? We can use the **Elbow Method**.

We calculate the **Inertia** (Within-Cluster Sum of Squares) for different values of $K$. Inertia measures how compact the clusters are. We want low inertia, but not at the cost of too many clusters.

We look for the "elbow" point where the inertia starts decreasing more slowly.

![Elbow Method](https://upload.wikimedia.org/wikipedia/commons/c/cd/DataClustering_ElbowCriterion.JPG)
*(Source: Wikimedia Commons - Elbow Method showing the optimal K where the curve bends)*

In [ ]:
inertia_list = []
k_values = range(1, 11)

for k in k_values:
    kmeans_model = KMeans(n_clusters=k, random_state=42)
    kmeans_model.fit(X_scaled)
    inertia_list.append(kmeans_model.inertia_)

# Plotting the Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(k_values, inertia_list, marker='o')
plt.title('Elbow Method For Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

**Interpretation**: Look for the point where the curve bends effectively. For this dataset, the elbow is likely around $K=3$, confirming our initial choice.

## 5. Hierarchical Clustering

Hierarchical clustering builds a hierarchy of clusters. It doesn't require pre-specifying $K$.

### Concepts
- **Agglomerative**: Start with each point as a cluster and merge the closest pairs efficiently.
- **Dendrogram**: A tree-like diagram that records the merges.

![Hierarchical Clustering Dendrogram](https://upload.wikimedia.org/wikipedia/commons/a/ad/Hierarchical_clustering_simple_diagram.svg)
*(Source: Wikimedia Commons - Dendrogram showing hierarchical cluster merging)*

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

# Calculate the linkage matrix using 'ward' linkage (minimizes variance)
mergings = linkage(X_scaled, method='ward')

# Plot the dendrogram
plt.figure(figsize=(12, 6))
dendrogram(mergings,
           labels=df['Cluster'].values, # Optional: using cluster labels from K-means as leaf labels
           leaf_rotation=90,
           leaf_font_size=6,
           truncate_mode='level', # Truncate to keep the plot readable if too many points
           p=5)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample index or Cluster Size')
plt.ylabel('Distance')
plt.show()

## 6. Exercises

Now it's your turn to practice using the classic **Iris dataset**.

**Task**: Cluster the iris flowers to identify different species based on their petal and sepal measurements.

1.  **Load the data**: Use `sklearn.datasets.load_iris`.
2.  **Preprocess**: Scale the features using `StandardScaler`.
3.  **Find K**: Use the Elbow Method to determine the optimal number of clusters (Hint: There are 3 species, so look for K=3).
4.  **Cluster**: Train a K-Means model with your chosen $K$.
5.  **Visualize**: Create a scatter plot of two features (e.g., Sepal Length vs Petal Length) colored by your cluster labels.